In [4]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END
from typing import TypedDict
from dotenv import load_dotenv
import os

C:\Users\praju\AppData\Local\Temp\ipykernel_31328\1863038152.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


In [5]:
def doc_loader(path):
    try:
        loader = DirectoryLoader(path,
                                glob="**/*.pdf",
                                loader_cls=PyMuPDFLoader,
                                show_progress=True)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {path}")
        return documents
    except Exception as e:
        print(f"Error loading documents from {path}: {e}")
        return None

def text_splitter(documents):
    print("Splitting documents into chunks...")
    try:
        if not documents:
            raise ValueError("No documents to split")
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                              chunk_overlap=150,
                                              length_function=len,
                                              separators=["\n\n", "\n", " ", ""])
        chunks = splitter.split_documents(documents)
        print(f"""Split into {len(chunks)} chunks
            Document splitting complete""")
        return chunks
    except Exception as e:
        print(f"Error splitting documents: {e}")
        return None

def create_vector(chunks):
    print('Loading embedding model...')
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    print('Creating vector store...')
    vector_store = Chroma.from_documents(chunks, 
                                         embedding, 
                                         collection_name="pdf_docs")
    print('Vector store created successfully')
    return vector_store

def llm_model():
    print('Loading LLM model...')
    llm = ChatOpenAI(
        model="moonshotai/kimi-k2.6:free", 
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1")
    return llm

def web_search(state):
    search_tool = TavilySearch(
        max_results=5
    )

### using the above components to build Agentic RAG pipeline

In [6]:
# defining graph state
class agentstate(TypedDict):
    question:str
    rewritten_query:str
    documents:list
    web_result:str
    answer:str

### Re-writing the user query  

In [7]:
def rewrite(state):
    query = state["question"]
    llm = llm_model()
    prompt = f"""
    You are a query rewriting assistant.

    Rewrite the query only if it improves retrieval quality.

    Rules:
    - Preserve meaning
    - Do not add information
    - Return the original query if already clear

    Query:
    {query}
    """
    rewritten_query = llm.invoke(prompt)
    return {
        "rewritten_query":rewritten_query.content
    }


### Retrival Block

In [8]:
def retriever(state):
    documents = doc_loader("../data")
    chunks = text_splitter(documents)
    vector_db = create_vector(chunks)
    query = state["question"]
    retriever_db = vector_db.as_retriever()
    r_docs = retriever_db.invoke(query) 
    return {
        r_docs
    }

### Grade Retrived documents


In [9]:
def web_search(state):
    query = state["question"]
    search_tool = web_search()
    results = search_tool.invoke(query)

    return {
        "web-results":results
    }

def grade_docuemnts(state):
    docs = state["documents"]
    if len(docs) > 0:
        return "Generate"
    else :
        return "web-search"

In [10]:
def generator(state):
    query = state["question"]
    docs = state.get("docuemnts",[])
    web_results = state.get("web_results","")

    if docs:
        context = "/n/n".join(
            [doc.page_content for doc in docs]
        )
    else:
        context = web_results

    prompt = f"""
Answer the following using the provided context

question:{query}

context:{context}
"""
    llm = llm_model()
    response = llm.invoke(prompt)

    return {
        "answer":response.content
    }

In [12]:
workflow = StateGraph(agentstate)

workflow.add_node(
    "rewrite",
    rewrite
)

workflow.add_node(
    "retriever",
    retriever
)

workflow.add_node(
    "web_search",
    web_search
)

workflow.add_node(
    "generate",
    generator
)

In [ ]:
workflow.set_entry_point("rewrite")

workflow.add_edge(
    "rewrite",
    "retriever"
)

workflow.add_conditional_edges(
    "retriever",
    grade_docuemnts,
    {
        "generate":"generate",
        "web_search":"web_search"
    }
)

workflow.add_edge(
    "web_search",
    "generate"
)

workflow.add_edge(
    "generate",
    END
)

graph = workflow.compile()